# 🌐 GlobalChain Crisis Lab
## Practical Assignment: Risk Management & Resilience in Logistics & Supply Chains

---

### 📖 Scenario
You are a **Supply Chain Risk Analyst** at **GlobalChain Ltd** — a multinational electronics
manufacturer. The company sources components from **25 suppliers** across 10 countries,
operates **4 factories**, and delivers through **6 distribution centres** worldwide.

**This year, the following crisis events are unfolding:**
- 🌊 A major typhoon has disrupted ports in Southeast Asia
- 🏭 A key semiconductor supplier in Taiwan faces production halts
- 🚢 A port strike in Europe is delaying shipments by 3–6 weeks
- 📈 Post-pandemic demand volatility is making forecasting extremely difficult

**Your mission:** Analyse the network, profile suppliers by risk, forecast uncertain demand,
simulate disruption scenarios, and produce Power BI dashboards for the board.

---

### 🎯 Learning Objectives
| Part | Topic | Key Tools |
|------|-------|-----------|
| A | Supply Chain Network & Vulnerability | NetworkX, centrality metrics |
| B | Supplier Risk Profiling | KMeans, PCA, sklearn |
| C | Risk Scoring & Heatmaps | matplotlib, risk matrix |
| D | Demand Forecasting & Uncertainty | ARIMA, Monte Carlo |
| E | Disruption Simulation | Monte Carlo, VaR, CVaR |
| F | Power BI Export | pandas, openpyxl |

---
> **Instructions:** Run cells in order. All `# TODO` sections are yours to complete.


In [ ]:
# ════════════════════════════════════════════════════
# 🔧  SETUP — Run this cell first
# ════════════════════════════════════════════════════
# Uncomment if packages are missing:
# !pip install networkx statsmodels scikit-learn plotly openpyxl scipy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import networkx as nx
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import statsmodels.api as sm
from statsmodels.tsa.arima.model import ARIMA
from scipy import stats
import warnings, random

warnings.filterwarnings("ignore")
np.random.seed(42)
random.seed(42)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.family"] = "DejaVu Sans"
plt.style.use("seaborn-v0_8-whitegrid")

print("All packages loaded successfully!")
print(f"  NumPy {np.__version__} | Pandas {pd.__version__} | NetworkX {nx.__version__}")


---
##  Scenario Setup — GlobalChain Ltd Data

Run the cell below to generate all synthetic datasets for the lab.
These simulate realistic supply chain data you would encounter in industry.


In [ ]:
# ════════════════════════════════════════════════════
# 📦  DATASET GENERATION
# ════════════════════════════════════════════════════
np.random.seed(42)
N = 25

NAMES = [
    "Apex Electronics","BrightCore Metals","ChipTech Asia","DriveMax Components",
    "EcoFlex Plastics","FastLink Cables","GlobalParts Ltd","HorizonSemi",
    "IronClad Steel","JetForce Motors","KineticBoard","LumiTech Displays",
    "MegaFab Industries","NanoPrecision","OmegaChip","PrimeSeal Packaging",
    "QuantumWire","RapidCast Metals","SilkRoute Textiles","TechBase Korea",
    "UniCore Systems","VoltArc Power","WaveForm Circuits","XcelMold","ZenithParts"
]
COUNTRIES = [
    "China","Germany","Taiwan","USA","Vietnam","China","India","South Korea",
    "Brazil","Japan","Mexico","Taiwan","China","Germany","South Korea","USA",
    "India","Turkey","Vietnam","South Korea","USA","Germany","China","Mexico","Japan"
]
REGIONS = [
    "Asia","Europe","Asia","Americas","Asia","Asia","Asia","Asia","Americas",
    "Asia","Americas","Asia","Asia","Europe","Asia","Americas","Asia","Europe",
    "Asia","Asia","Americas","Europe","Asia","Americas","Asia"
]

# ── Supplier Master ────────────────────────────────────────────
suppliers_df = pd.DataFrame({
    "supplier_id":       [f"SUP-{i:03d}" for i in range(1, N+1)],
    "name":              NAMES,
    "country":           COUNTRIES,
    "region":            REGIONS,
    "annual_spend_musd": np.round(np.random.lognormal(2.0, 0.8, N), 2),
    "lead_time_days":    np.random.randint(7, 90, N),
    "otd_pct":           np.round(np.random.uniform(70, 99, N), 1),
    "reject_rate_pct":   np.round(np.random.uniform(0.1, 8.0, N), 2),
    "financial_score":   np.round(np.random.uniform(30, 95, N), 1),
    "geo_risk_score":    np.round(np.random.uniform(10, 85, N), 1),
    "single_source":     np.random.choice([True, False], N, p=[0.3, 0.7]),
    "has_backup":        np.random.choice([True, False], N, p=[0.4, 0.6]),
    "esg_score":         np.round(np.random.uniform(20, 90, N), 1),
    "capacity_util_pct": np.round(np.random.uniform(50, 98, N), 1),
    "years_relation":    np.random.randint(1, 20, N),
    "criticality":       np.random.choice(["Critical","High","Medium","Low"], N,
                                           p=[0.2, 0.3, 0.3, 0.2])
})

# ── Network Nodes ──────────────────────────────────────────────
nodes_df = pd.DataFrame({
    "node_id": (["S1","S2","S3","S4","S5"] +
                ["F1","F2","F3","F4"] +
                ["DC1","DC2","DC3","DC4","DC5","DC6"] +
                ["C1","C2","C3","C4","C5"]),
    "type": (["Supplier"]*5 + ["Factory"]*4 +
             ["DC"]*6 + ["Customer"]*5),
    "location": (["China","Taiwan","Germany","Vietnam","India",
                  "Germany","USA","China","Mexico",
                  "UK","France","USA","UAE","Singapore","Brazil",
                  "Europe","North America","Asia","Middle East","Latin America"]),
    "criticality": ([0.9,0.95,0.7,0.6,0.65,
                     0.85,0.80,0.90,0.75,
                     0.5,0.6,0.7,0.55,0.65,0.5,
                     0.4,0.5,0.6,0.4,0.45])
})

# ── Network Edges ──────────────────────────────────────────────
EDGES = [
    ("S1","F1",{"weight":100,"risk":0.3}), ("S1","F3",{"weight":80,"risk":0.3}),
    ("S2","F1",{"weight":150,"risk":0.5}), ("S2","F3",{"weight":90,"risk":0.5}),
    ("S3","F2",{"weight":120,"risk":0.2}), ("S3","F1",{"weight":60,"risk":0.2}),
    ("S4","F4",{"weight":70,"risk":0.4}),  ("S4","F2",{"weight":50,"risk":0.4}),
    ("S5","F3",{"weight":80,"risk":0.35}), ("S5","F4",{"weight":40,"risk":0.35}),
    ("F1","DC1",{"weight":200,"risk":0.15}),("F1","DC2",{"weight":150,"risk":0.2}),
    ("F2","DC3",{"weight":180,"risk":0.1}), ("F2","DC4",{"weight":130,"risk":0.25}),
    ("F3","DC1",{"weight":90,"risk":0.3}),  ("F3","DC5",{"weight":110,"risk":0.2}),
    ("F4","DC4",{"weight":100,"risk":0.15}),("F4","DC6",{"weight":85,"risk":0.35}),
    ("DC1","C1",{"weight":250,"risk":0.1}), ("DC2","C1",{"weight":100,"risk":0.1}),
    ("DC3","C2",{"weight":200,"risk":0.1}), ("DC4","C3",{"weight":180,"risk":0.1}),
    ("DC5","C4",{"weight":140,"risk":0.15}),("DC6","C5",{"weight":90,"risk":0.2}),
]

# ── 36-Month Demand Time Series ────────────────────────────────
months = pd.date_range("2022-01-01", periods=36, freq="MS")
trend  = np.linspace(1000, 1350, 36)
seasonal = 150 * np.sin(2 * np.pi * np.arange(36) / 12)
disruption = np.zeros(36)
disruption[18:21] = -200
disruption[28:30] = -150
noise = np.random.normal(0, 60, 36)
demand_series = trend + seasonal + disruption + noise

demand_df = pd.DataFrame({
    "date":   months,
    "demand": np.round(demand_series, 0),
    "month":  months.month,
    "year":   months.year
})

# ── Risk Register ──────────────────────────────────────────────
risk_register = pd.DataFrame({
    "risk_id":   [f"R{i:02d}" for i in range(1, 16)],
    "risk_name": [
        "Port Strike (Europe)","Typhoon (SE Asia)","Semiconductor Shortage",
        "Geopolitical Tension (Taiwan)","Supplier Bankruptcy","Cyber Attack on ERP",
        "Pandemic Resurgence","Raw Material Price Spike","Regulatory Change",
        "Quality Failure (Critical Part)","Natural Disaster (Japan)",
        "Currency Fluctuation","Logistics Capacity Shortage","Labour Strike",
        "IT System Failure"
    ],
    "category": [
        "External","Natural","Supply","Geopolitical","Supplier","Cyber",
        "Health","Market","Regulatory","Quality","Natural",
        "Financial","Logistics","Labour","Technology"
    ],
    "likelihood": [4,3,5,3,2,3,2,4,3,3,2,4,3,2,3],
    "impact":     [4,5,5,5,4,4,5,3,3,5,4,2,3,3,3],
    "control":    [
        "Dual sourcing","Insurance","Strategic stock","Diversification",
        "Credit checks","Firewall + DR","Contingency plan","Hedging",
        "Legal monitoring","QMS","BCM plan","Natural hedge","Spot market",
        "Labour agreements","Redundancy"
    ]
})
risk_register["risk_score"] = risk_register["likelihood"] * risk_register["impact"]
risk_register["priority"] = pd.cut(
    risk_register["risk_score"], bins=[0,6,12,25],
    labels=["Low","Medium","High"])

print("All datasets generated!")
print(f"  suppliers_df : {suppliers_df.shape}")
print(f"  nodes_df     : {nodes_df.shape}   ({len(EDGES)} edges)")
print(f"  demand_df    : {demand_df.shape} (36 months)")
print(f"  risk_register: {risk_register.shape}")
display(suppliers_df.head(3))


---
## PART A — Supply Chain Network & Vulnerability Analysis

A supply chain is a **network** — and like any network, some nodes are critical
bottlenecks whose failure cascades into major disruption (Fukushima 2011, Suez 2021).

**NetworkX** lets us model the supply chain as a directed graph and apply graph theory
to find vulnerabilities.

| Metric | Meaning in Supply Chain |
|--------|-------------------------|
| **Betweenness Centrality** | How often a node lies on the shortest path between others — **the most important bottleneck measure** |
| **Degree Centrality** | How many connections a node has |
| **PageRank** | Importance weighted by the importance of neighbours |

> High betweenness = CRITICAL VULNERABILITY: if this node fails, many paths break.


In [ ]:
# ════════════════════════════════════════════════════
#  PART A.1 — Build Network & Compute Centrality
# ════════════════════════════════════════════════════

G = nx.DiGraph()

for _, row in nodes_df.iterrows():
    G.add_node(row["node_id"], node_type=row["type"],
               location=row["location"], criticality=row["criticality"])

for u, v, attrs in EDGES:
    G.add_edge(u, v, **attrs)

betweenness  = nx.betweenness_centrality(G, normalized=True, weight="weight")
in_deg_c     = nx.in_degree_centrality(G)
out_deg_c    = nx.out_degree_centrality(G)
pagerank     = nx.pagerank(G, alpha=0.85, weight="weight")

centrality_df = pd.DataFrame({
    "node_id":    list(betweenness.keys()),
    "type":       [G.nodes[n]["node_type"] for n in betweenness],
    "betweenness":[round(v, 4) for v in betweenness.values()],
    "in_degree":  [round(in_deg_c[n], 4) for n in betweenness],
    "out_degree": [round(out_deg_c[n], 4) for n in betweenness],
    "pagerank":   [round(pagerank[n], 4) for n in betweenness],
    "criticality":[G.nodes[n]["criticality"] for n in betweenness],
}).sort_values("betweenness", ascending=False).reset_index(drop=True)

print("TOP 10 CRITICAL NODES (Betweenness Centrality):")
display(centrality_df.head(10))


In [ ]:
# ════════════════════════════════════════════════════
#   PART A.2 — Visualise Network + Resilience Curves
# ════════════════════════════════════════════════════

COLOUR_MAP = {"Supplier":"#E74C3C","Factory":"#E67E22","DC":"#3498DB","Customer":"#2ECC71"}
node_col  = [COLOUR_MAP[G.nodes[n]["node_type"]] for n in G.nodes()]
node_size = [3500 * betweenness[n] + 400 for n in G.nodes()]
pos = nx.spring_layout(G, seed=42, k=2.5)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# — Left: Network diagram ——————————————————————————
ax = axes[0]
nx.draw_networkx_edges(G, pos, ax=ax, edge_color="#BDC3C7",
                       arrows=True, arrowsize=15, width=1.5, alpha=0.6)
nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_col,
                       node_size=node_size, alpha=0.92)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=7, font_weight="bold")
ax.set_title("Supply Chain Network\n(node size = betweenness centrality)",
             fontsize=13, fontweight="bold")
ax.axis("off")
for nt, c in COLOUR_MAP.items():
    ax.scatter([], [], c=c, s=100, label=nt)
ax.legend(loc="upper left", fontsize=9)

# — Right: Top-10 betweenness bar ——————————————————
ax2 = axes[1]
top10 = centrality_df.head(10)
bar_col = [COLOUR_MAP[t] for t in top10["type"]]
bars = ax2.barh(top10["node_id"], top10["betweenness"],
                color=bar_col, edgecolor="white", height=0.7)
ax2.set_xlabel("Betweenness Centrality", fontsize=11)
ax2.set_title("Top 10 Critical Nodes", fontsize=13, fontweight="bold")
ax2.invert_yaxis()
for bar, val in zip(bars, top10["betweenness"]):
    ax2.text(bar.get_width() + 0.003, bar.get_y() + bar.get_height()/2,
             f"{val:.3f}", va="center", fontsize=9)

plt.tight_layout()
plt.savefig("network_graph.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: network_graph.png")


In [ ]:
# ════════════════════════════════════════════════════
#  PART A.3 — Node Failure Simulation
# ════════════════════════════════════════════════════
# Compare: targeted attack (remove highest-betweenness first)
# vs random failure  →  reveals how robust the network really is

def simulate_failures(graph, removal_order):
    # Remove nodes one by one, track largest connected component
    G2 = graph.copy()
    results = []
    for node in removal_order:
        if node in G2:
            G2.remove_node(node)
            ud = G2.to_undirected()
            comps = list(nx.connected_components(ud))
            largest = max((len(c) for c in comps), default=0)
            results.append({
                "removed_node": node,
                "largest_component": largest,
                "n_components": len(comps),
                "nodes_remaining": G2.number_of_nodes()
            })
    return pd.DataFrame(results)

targeted_order = centrality_df.sort_values("betweenness", ascending=False)["node_id"].tolist()
random_order   = targeted_order.copy()
random.shuffle(random_order)

targeted_res = simulate_failures(G, targeted_order)
random_res   = simulate_failures(G, random_order)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(targeted_res.index, targeted_res["largest_component"],
        "r-o", lw=2.5, ms=6, label="Targeted Attack (high betweenness first)")
ax.plot(random_res.index, random_res["largest_component"],
        "b--s", lw=2.5, ms=6, label="Random Failures")
ax.fill_between(targeted_res.index,
                targeted_res["largest_component"],
                random_res["largest_component"],
                alpha=0.12, color="red", label="Vulnerability gap")
ax.set_xlabel("Number of Nodes Removed", fontsize=12)
ax.set_ylabel("Largest Connected Component Size", fontsize=12)
ax.set_title("Network Resilience: Targeted Attack vs Random Failure",
             fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig("resilience_curve.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: resilience_curve.png")
print(f"\nKey insight: Removing top-3 nodes drops network by "
      f"{targeted_res['largest_component'].iloc[0] - targeted_res['largest_component'].iloc[2]} nodes "
      f"vs {random_res['largest_component'].iloc[0] - random_res['largest_component'].iloc[2]} for random")


### Task A — Your Turn!

1. **Identify the top 3 most critical nodes.** Explain *why* they are critical using
   the centrality metrics (betweenness, pagerank, in/out-degree).

2. **The typhoon has knocked out node `F3`** (China factory). Write code that:
   - Removes `F3` from the graph
   - Finds which customers (`C1–C5`) have **lost all supply paths**
   - Calculates what % of total edge weight (volume) is now unreachable

3. **Interpret the resilience gap chart.** Write 2–3 sentences for a board presentation
   explaining what the gap means for GlobalChain's risk strategy.

4. *(Bonus)* Add a **weighted resilience metric** that accounts for edge `weight`
   (shipment volume), not just node count.

```python
# TODO — Task A code here
# Hint for Q2:
# G_test = G.copy()
# G_test.remove_node("F3")
# for c in ["C1","C2","C3","C4","C5"]:
#     reachable = nx.has_path(G_test, "S1", c)  # check multiple source nodes too
```


---
##  PART B — Supplier Risk Profiling & ML Clustering

With 25 suppliers across 10 countries, manually categorising risks is slow and biased.
We use **K-Means clustering** to group suppliers into **risk tiers** based on multiple
risk dimensions simultaneously.

### Risk Dimensions:
- **Operational:** delivery rate, lead time, reject rate
- **Financial:** financial stability, spend exposure
- **Strategic:** single sourcing, backup availability
- **Geopolitical/ESG:** country risk, ESG score


In [ ]:
# ════════════════════════════════════════════════════
#  PART B.1 — Compute Composite Risk Score
# ════════════════════════════════════════════════════

df = suppliers_df.copy()

# Flip scores so that higher always means MORE risky
df["delivery_risk"]    = 100 - df["otd_pct"]
df["financial_risk"]   = 100 - df["financial_score"]
df["esg_risk"]         = 100 - df["esg_score"]
df["backup_risk"]      = (~df["has_backup"]).astype(int) * 50
df["sole_src_risk"]    = df["single_source"].astype(int) * 50

WEIGHTS = {
    "delivery_risk":   0.20,
    "financial_risk":  0.20,
    "geo_risk_score":  0.20,
    "reject_rate_pct": 0.10,
    "esg_risk":        0.10,
    "backup_risk":     0.10,
    "sole_src_risk":   0.10,
}

df["composite_risk"] = sum(df[col] * w for col, w in WEIGHTS.items()).round(2)

# K-Means clustering into 4 risk tiers
FEATURES = list(WEIGHTS.keys()) + ["lead_time_days", "capacity_util_pct"]
X = StandardScaler().fit_transform(df[FEATURES])

kmeans = KMeans(n_clusters=4, random_state=42, n_init=20)
df["cluster"] = kmeans.fit_predict(X)

cluster_means = df.groupby("cluster")["composite_risk"].mean().sort_values()
label_map     = {old: new for new, old in enumerate(cluster_means.index)}
df["tier_num"]  = df["cluster"].map(label_map)
TIER_LABELS = {0:"Tier 1 - Low Risk",1:"Tier 2 - Medium Risk",
               2:"Tier 3 - High Risk",3:"Tier 4 - Critical Risk"}
df["risk_tier"] = df["tier_num"].map(TIER_LABELS)

print("Risk Tier Distribution:")
print(df["risk_tier"].value_counts().sort_index().to_string())
print(f"\nComposite risk stats by tier:")
display(df.groupby("risk_tier")["composite_risk"].agg(["mean","min","max"]).round(2))


In [ ]:
# ════════════════════════════════════════════════════
#   PART B.2 — Visualise Clusters & Risk Ranking
# ════════════════════════════════════════════════════

TIER_COLOURS = {0:"#2ECC71", 1:"#F1C40F", 2:"#E67E22", 3:"#E74C3C"}

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)
df["pca1"] = X_pca[:, 0]
df["pca2"] = X_pca[:, 1]

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# — PCA Cluster Scatter ————————————————————————————
ax = axes[0]
for tier, grp in df.groupby("tier_num"):
    ax.scatter(grp["pca1"], grp["pca2"],
               c=TIER_COLOURS[tier], s=80, alpha=0.85,
               edgecolors="white", lw=0.5, label=TIER_LABELS[tier])
    for _, row in grp.iterrows():
        ax.annotate(row["supplier_id"],
                    (row["pca1"], row["pca2"]),
                    fontsize=6.5, ha="center", va="bottom")
ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.0%} var)", fontsize=11)
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.0%} var)", fontsize=11)
ax.set_title("Supplier Risk Clusters (PCA projection)",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=9)

# — Composite Risk Ranking ————————————————————————
ax2 = axes[1]
sdf = df.sort_values("composite_risk", ascending=True)
bar_col = [TIER_COLOURS[t] for t in sdf["tier_num"]]
ax2.barh(sdf["name"], sdf["composite_risk"],
         color=bar_col, edgecolor="white", height=0.72)
ax2.axvline(50, color="red", lw=1.8, linestyle="--", alpha=0.7, label="Risk threshold (50)")
ax2.set_xlabel("Composite Risk Score", fontsize=11)
ax2.set_title("Supplier Risk Ranking", fontsize=13, fontweight="bold")
ax2.tick_params(axis="y", labelsize=7.5)
ax2.legend(fontsize=10)

plt.suptitle("Supplier Risk Profiling — K-Means (4 Tiers)",
             fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("supplier_clustering.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: supplier_clustering.png")
print(f"\nHighest risk supplier : {df.loc[df['composite_risk'].idxmax(), 'name']}  "
      f"(score {df['composite_risk'].max():.1f})")
print(f"Lowest  risk supplier : {df.loc[df['composite_risk'].idxmin(), 'name']}  "
      f"(score {df['composite_risk'].min():.1f})")


###  Task B — Your Turn!

1. **Redesign the weights** to model a scenario where GlobalChain is *most worried about
   geopolitical risk*. Re-run clustering and list which suppliers moved between tiers.

2. **Elbow Curve:** Run K-Means for k = 2 to 8. Plot the inertia (elbow curve) and
   justify the optimal number of risk tiers.

3. **Spend-at-risk:** Create a bar chart showing total `annual_spend_musd` at risk
   (Tier 3 + Tier 4 only), broken down by region.

```python
# TODO — Task B code here
# Hint for Elbow Curve:
# inertias = []
# for k in range(2, 9):
#     km = KMeans(n_clusters=k, random_state=42, n_init=10)
#     km.fit(X)
#     inertias.append(km.inertia_)
# plt.plot(range(2,9), inertias, "o-")
```


---
##  PART C — Risk Scoring & Heatmaps

The **5×5 Risk Matrix** (Likelihood × Impact) is the standard tool for communicating
risk priorities to executives. We will build a colour-coded heatmap and bubble chart
for GlobalChain's 15 identified risks.


In [ ]:
# ════════════════════════════════════════════════════
#   PART C — Risk Heatmap & Bubble Chart
# ════════════════════════════════════════════════════

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# — Left: Classic 5×5 Risk Matrix ————————————————
ax = axes[0]
grid = np.array([
    [1,  2,  3,  4,  5],
    [2,  4,  6,  8,  10],
    [3,  6,  9,  12, 15],
    [4,  8,  12, 16, 20],
    [5,  10, 15, 20, 25]
])
cmap_risk = mcolors.LinearSegmentedColormap.from_list(
    "risk", ["#2ECC71","#F1C40F","#E67E22","#E74C3C"], N=256)
ax.imshow(grid, cmap=cmap_risk, aspect="auto", origin="lower",
          extent=[0.5,5.5,0.5,5.5], alpha=0.65, vmin=1, vmax=25)

for i in range(6):
    ax.axhline(i+0.5, color="white", lw=1.5)
    ax.axvline(i+0.5, color="white", lw=1.5)

np.random.seed(99)
for _, row in risk_register.iterrows():
    jx = np.random.uniform(-0.12, 0.12)
    jy = np.random.uniform(-0.12, 0.12)
    ax.scatter(row["impact"]+jx, row["likelihood"]+jy,
               s=130, zorder=5, color="#2C3E50", alpha=0.92)
    ax.annotate(row["risk_id"],
                (row["impact"]+jx, row["likelihood"]+jy),
                fontsize=7, ha="center", va="center",
                color="white", fontweight="bold")

ax.set_xlabel("Impact  (1=Negligible → 5=Catastrophic)", fontsize=11)
ax.set_ylabel("Likelihood  (1=Rare → 5=Almost Certain)", fontsize=11)
ax.set_title("GlobalChain Risk Matrix (5×5)", fontsize=13, fontweight="bold")
ax.set_xticks([1,2,3,4,5])
ax.set_yticks([1,2,3,4,5])
ax.set_xticklabels(["Negligible","Minor","Moderate","Major","Catastrophic"], fontsize=8.5)
ax.set_yticklabels(["Rare","Unlikely","Possible","Likely","Almost Certain"], fontsize=8.5)
ax.text(4.5, 4.6, "HIGH", fontsize=10, color="white", alpha=0.85,
        fontweight="bold", ha="center")
ax.text(1.5, 1.4, "LOW", fontsize=10, color="white", alpha=0.7, ha="center")

# — Right: Bubble Chart ———————————————————————————
ax2 = axes[1]
CAT_COL = {
    "External":"#E74C3C","Natural":"#3498DB","Supply":"#E67E22",
    "Geopolitical":"#9B59B6","Supplier":"#E74C3C","Cyber":"#1ABC9C",
    "Health":"#F39C12","Market":"#2ECC71","Regulatory":"#95A5A6",
    "Quality":"#D35400","Financial":"#27AE60","Logistics":"#2980B9",
    "Labour":"#8E44AD","Technology":"#16A085"
}
for _, row in risk_register.iterrows():
    col = CAT_COL.get(row["category"], "#7F8C8D")
    ax2.scatter(row["impact"], row["likelihood"],
                s=row["risk_score"]*32, color=col, alpha=0.72,
                edgecolors="white", lw=1.5, zorder=3)
    ax2.annotate(row["risk_name"],
                 (row["impact"], row["likelihood"]),
                 fontsize=7.5, ha="center", va="bottom",
                 xytext=(0, 10), textcoords="offset points")
ax2.set_xlabel("Impact", fontsize=11)
ax2.set_ylabel("Likelihood", fontsize=11)
ax2.set_xlim(0.5, 5.8)
ax2.set_ylim(0.5, 5.5)
ax2.set_xticks([1,2,3,4,5])
ax2.set_yticks([1,2,3,4,5])
ax2.set_title("Risk Bubble Chart (bubble size = risk score)",
              fontsize=13, fontweight="bold")

plt.tight_layout()
plt.savefig("risk_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: risk_heatmap.png")
print("\nTop 5 Risks:")
display(risk_register.sort_values("risk_score", ascending=False)
        [["risk_name","category","likelihood","impact","risk_score","priority","control"]]
        .head(5).reset_index(drop=True))


###  Task C — Your Turn!

1. **Add 3 new risks** relevant to the scenario (port strike aftermath, supplier
   concentration in Asia, demand forecast error). Re-plot and explain where they fall.

2. **Mitigation simulation:** For the **top 3 risks by score**, design a mitigation
   that reduces likelihood OR impact by 1 point each.
   Plot the "before vs after" positions on the matrix.

3. **Category analysis:** Create a grouped bar chart of **average risk score by category**.
   Which category poses the greatest overall threat?

```python
# TODO — Task C code here
```


---
##  PART D — Demand Forecasting & Uncertainty Quantification

Supply chain resilience requires **anticipating demand under uncertainty**.
We use **ARIMA** for time series forecasting and **Monte Carlo simulation**
to generate a range of scenarios — from optimistic to catastrophic.

> *"All models are wrong, but some are useful." — George Box*


In [ ]:
# ════════════════════════════════════════════════════
#   PART D.1 — Demand Decomposition
# ════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# — Demand Time Series ————————————————————————————
ax = axes[0,0]
ax.plot(demand_df["date"], demand_df["demand"], "b-o", ms=4, lw=2, label="Actual")
ax.fill_between(demand_df["date"], demand_df["demand"]-80, demand_df["demand"]+80,
                alpha=0.12, color="blue")
ax.axvspan(pd.to_datetime("2023-07-01"), pd.to_datetime("2023-09-30"),
           alpha=0.15, color="red", label="Disruption 1 (typhoon)")
ax.axvspan(pd.to_datetime("2024-05-01"), pd.to_datetime("2024-06-30"),
           alpha=0.12, color="orange", label="Disruption 2 (port strike)")
ax.set_title("36-Month Demand History", fontsize=12, fontweight="bold")
ax.set_xlabel("Date"); ax.set_ylabel("Demand (units)")
ax.legend(fontsize=8)

# — Seasonal Decomposition ————————————————————————
decomp = sm.tsa.seasonal_decompose(demand_df["demand"], model="additive", period=12)

ax2 = axes[0,1]
ax2.plot(demand_df["date"], decomp.seasonal, "g-", lw=2)
ax2.axhline(0, color="black", lw=0.8, linestyle="--")
ax2.set_title("Seasonal Component", fontsize=12, fontweight="bold")
ax2.set_xlabel("Date"); ax2.set_ylabel("Seasonal Effect (units)")

ax3 = axes[1,0]
ax3.plot(demand_df["date"], decomp.trend, "r-", lw=2.5, label="Trend")
ax3.plot(demand_df["date"], demand_df["demand"], "b-", alpha=0.35, lw=1, label="Actual")
ax3.set_title("Trend Component", fontsize=12, fontweight="bold")
ax3.legend(fontsize=9)

ax4 = axes[1,1]
ax4.plot(demand_df["date"], decomp.resid, color="purple", lw=1.5, alpha=0.85)
ax4.fill_between(demand_df["date"], decomp.resid, 0,
                 where=(decomp.resid > 0), alpha=0.3, color="green", label="+ve shock")
ax4.fill_between(demand_df["date"], decomp.resid, 0,
                 where=(decomp.resid < 0), alpha=0.3, color="red", label="-ve shock")
ax4.axhline(0, color="black", lw=1, linestyle="--")
ax4.set_title("Residuals (unexplained volatility)", fontsize=12, fontweight="bold")
ax4.legend(fontsize=9)

plt.suptitle("GlobalChain Demand Decomposition", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("demand_decomposition.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ════════════════════════════════════════════════════
#  PART D.2 — ARIMA Forecast + Monte Carlo Fan
# ════════════════════════════════════════════════════

# Fit ARIMA(2,1,2)
arima_model = ARIMA(demand_df["demand"], order=(2, 1, 2))
arima_fit   = arima_model.fit()
print(arima_fit.summary().tables[0])

HORIZON = 12
forecast_obj  = arima_fit.get_forecast(steps=HORIZON)
forecast_mean = forecast_obj.predicted_mean
forecast_ci   = forecast_obj.conf_int(alpha=0.10)
forecast_dates = pd.date_range(
    start=demand_df["date"].iloc[-1] + pd.DateOffset(months=1),
    periods=HORIZON, freq="MS"
)

# Monte Carlo demand scenarios (5,000 paths)
N_SIM  = 5000
sigma  = np.std(decomp.resid.dropna())
mc_mat = np.zeros((N_SIM, HORIZON))

for i in range(N_SIM):
    shock_factor = np.random.choice([1.0, 0.75, 0.55], p=[0.65, 0.25, 0.10])
    mc_mat[i] = forecast_mean.values + np.random.normal(0, sigma * shock_factor, HORIZON)

p5  = np.percentile(mc_mat, 5,  axis=0)
p25 = np.percentile(mc_mat, 25, axis=0)
p75 = np.percentile(mc_mat, 75, axis=0)
p95 = np.percentile(mc_mat, 95, axis=0)

fig, ax = plt.subplots(figsize=(14, 6))
ax.plot(demand_df["date"], demand_df["demand"], "b-", lw=2.5,
        label="Historical Demand", zorder=4)

# MC fan
ax.fill_between(forecast_dates, p5, p95, alpha=0.15, color="orange",
                label="MC 90% range (5th–95th pct)")
ax.fill_between(forecast_dates, p25, p75, alpha=0.30, color="orange",
                label="MC 50% range (25th–75th pct)")

# ARIMA line + CI
ax.plot(forecast_dates, forecast_mean, "r--", lw=2.5, label="ARIMA mean forecast")
ax.fill_between(forecast_dates, forecast_ci.iloc[:,0], forecast_ci.iloc[:,1],
                alpha=0.20, color="red", label="ARIMA 90% CI")

# Sample MC paths
for i in range(120):
    ax.plot(forecast_dates, mc_mat[i], "gray", alpha=0.04, lw=0.8)

ax.axvline(demand_df["date"].iloc[-1], color="black", lw=1.5,
           linestyle="--", label="Forecast start")
ax.set_xlabel("Date", fontsize=12); ax.set_ylabel("Demand (units)", fontsize=12)
ax.set_title("12-Month Demand Forecast with Monte Carlo Uncertainty (5,000 simulations)",
             fontsize=13, fontweight="bold")
ax.legend(fontsize=9, loc="upper left")
plt.tight_layout()
plt.savefig("demand_forecast.png", dpi=150, bbox_inches="tight")
plt.show()

forecast_summary = pd.DataFrame({
    "Date": forecast_dates,
    "ARIMA_Mean": forecast_mean.round(0).astype(int),
    "MC_P5":  p5.round(0).astype(int),
    "MC_P25": p25.round(0).astype(int),
    "MC_P75": p75.round(0).astype(int),
    "MC_P95": p95.round(0).astype(int),
})
print("\n12-Month Forecast Summary:")
display(forecast_summary)


###  Task D — Your Turn!

1. **Black Swan scenario:** Add a 4th Monte Carlo shock: demand drops 40% for 3 months
   (simulating a factory shutdown). Give it 5% probability.
   How does this change the P5 percentile forecast?

2. **Safety stock calculation:** Using the Monte Carlo output, compute the **safety stock**
   needed to achieve a 95% service level.

   *Formula:*  `Safety Stock = z * σ_demand * sqrt(lead_time_months)`
   where `z = scipy.stats.norm.ppf(0.95)` ≈ 1.645

3. **Model comparison:** Try Exponential Smoothing or a different ARIMA(p,d,q) order.
   Use the last 6 months as a holdout set and compare RMSE.

```python
# TODO — Task D code here
# Hint Q2:
# z = stats.norm.ppf(0.95)
# sigma_d = np.std(mc_mat, axis=0).mean()
# avg_lt  = suppliers_df["lead_time_days"].mean() / 30  # months
# safety_stock = z * sigma_d * np.sqrt(avg_lt)
```


---
##  PART E — Monte Carlo Disruption Simulation

Now we bring **everything together**: a full supply chain disruption simulator.

For each of **10,000 scenarios**, we randomly sample which disruption events occur,
their duration and magnitude, and calculate the **financial loss** and
**service level degradation** for GlobalChain.

> **VaR (Value at Risk):** The loss that will not be exceeded in X% of scenarios.
> **CVaR (Expected Shortfall):** The average loss in the worst X% of scenarios.


In [ ]:
# ════════════════════════════════════════════════════
# ⚡  PART E.1 — Run Monte Carlo Disruption Simulation
# ════════════════════════════════════════════════════

N_SCENARIOS    = 10_000
ANNUAL_REVENUE = 500_000_000   # $500M
DAILY_REVENUE  = ANNUAL_REVENUE / 365
np.random.seed(42)

# Disruption event library (calibrated to industry data)
EVENTS = [
    {"name":"Port Strike",          "prob":0.25, "min_d":10, "max_d":45,
     "impact_range":(0.05,0.20)},
    {"name":"Supplier Bankruptcy",  "prob":0.10, "min_d":30, "max_d":120,
     "impact_range":(0.08,0.35)},
    {"name":"Natural Disaster",     "prob":0.08, "min_d":7,  "max_d":60,
     "impact_range":(0.03,0.25)},
    {"name":"Cyber Attack",         "prob":0.15, "min_d":3,  "max_d":21,
     "impact_range":(0.02,0.15)},
    {"name":"Demand Shock",         "prob":0.30, "min_d":14, "max_d":90,
     "impact_range":(0.05,0.30)},
    {"name":"Geopolitical Tension", "prob":0.12, "min_d":30, "max_d":180,
     "impact_range":(0.05,0.25)},
]

sim_results = []
for sid in range(N_SCENARIOS):
    total_loss = 0.0
    service_hit = 0.0
    events_hit = []

    for ev in EVENTS:
        if np.random.random() < ev["prob"]:
            duration   = np.random.randint(ev["min_d"], ev["max_d"]+1)
            impact_pct = np.random.uniform(*ev["impact_range"])
            total_loss += DAILY_REVENUE * impact_pct * duration
            service_hit = max(service_hit, impact_pct * np.random.uniform(0.5, 1.5))
            events_hit.append(ev["name"])

    sim_results.append({
        "scenario_id":     sid,
        "n_events":        len(events_hit),
        "total_loss_usd":  total_loss,
        "loss_pct_rev":    total_loss / ANNUAL_REVENUE * 100,
        "service_level":   max(0.0, 1.0 - service_hit) * 100,
        "events":          ", ".join(events_hit) if events_hit else "None",
        "disrupted":       len(events_hit) > 0,
    })

sim_df = pd.DataFrame(sim_results)

var95  = np.percentile(sim_df["total_loss_usd"], 95) / 1e6
var99  = np.percentile(sim_df["total_loss_usd"], 99) / 1e6
cvar95 = sim_df[sim_df["total_loss_usd"] >= np.percentile(
         sim_df["total_loss_usd"], 95)]["total_loss_usd"].mean() / 1e6

print(f"Simulated {N_SCENARIOS:,} scenarios")
print(f"\nRISK METRICS SUMMARY")
print(f"  Expected Annual Loss : ${sim_df['total_loss_usd'].mean()/1e6:.2f}M")
print(f"  VaR 95%              : ${var95:.2f}M")
print(f"  VaR 99%              : ${var99:.2f}M")
print(f"  CVaR 95%             : ${cvar95:.2f}M")
print(f"  P(service < 95%)     : {(sim_df['service_level'] < 95).mean():.1%}")
print(f"  P(any disruption)    : {sim_df['disrupted'].mean():.1%}")


In [ ]:
# ════════════════════════════════════════════════════
# ⚡  PART E.2 — Visualise Simulation Results
# ════════════════════════════════════════════════════

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# — Loss Distribution ——————————————————————————————
ax = axes[0,0]
losses_m = sim_df["total_loss_usd"] / 1e6
ax.hist(losses_m[losses_m > 0], bins=80, color="#E74C3C",
        alpha=0.72, edgecolor="white", lw=0.4)
for pct, col, lbl in [(95,"#E67E22","VaR 95%"),(99,"#922B21","VaR 99%")]:
    val = np.percentile(sim_df["total_loss_usd"], pct) / 1e6
    ax.axvline(val, color=col, lw=2.2, linestyle="--", label=f"{lbl}: ${val:.1f}M")
ax.set_xlabel("Annual Loss ($M)", fontsize=11)
ax.set_ylabel("Frequency", fontsize=11)
ax.set_title("Distribution of Annual Disruption Losses\n(10,000 Monte Carlo scenarios)",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=10)

# — Loss Exceedance Curve ——————————————————————————
ax2 = axes[0,1]
sl = np.sort(sim_df["total_loss_usd"].values)[::-1] / 1e6
ep = np.arange(1, len(sl)+1) / len(sl) * 100
ax2.plot(sl, ep, "b-", lw=2.2)
ax2.fill_between(sl, ep, alpha=0.1, color="blue")
ax2.axhline(5, color="red",    lw=1.5, linestyle="--", label="5%  (1-in-20 yr event)")
ax2.axhline(1, color="orange", lw=1.5, linestyle="--", label="1%  (1-in-100 yr event)")
ax2.set_xlabel("Annual Loss ($M)", fontsize=11)
ax2.set_ylabel("Prob. of Exceedance (%)", fontsize=11)
ax2.set_title("Loss Exceedance Probability Curve", fontsize=12, fontweight="bold")
ax2.legend(fontsize=9)
ax2.set_xlim(left=0)

# — Service Level Distribution ————————————————————
ax3 = axes[1,0]
ax3.hist(sim_df["service_level"], bins=60, color="#3498DB",
         alpha=0.75, edgecolor="white", lw=0.4)
ax3.axvline(sim_df["service_level"].mean(), color="#E74C3C", lw=2.5,
            linestyle="--", label=f"Mean: {sim_df['service_level'].mean():.1f}%")
ax3.axvline(95, color="green", lw=2.2, linestyle=":",
            label="Target: 95%")
ax3.set_xlabel("Service Level (%)", fontsize=11)
ax3.set_ylabel("Frequency", fontsize=11)
ax3.set_title("Service Level Distribution", fontsize=12, fontweight="bold")
ax3.legend(fontsize=10)

# — Event Impact Bar Chart ————————————————————————
ax4 = axes[1,1]
ev_loss = {}
for _, row in sim_df.iterrows():
    if row["events"] != "None":
        for ev in row["events"].split(", "):
            ev_loss.setdefault(ev, []).append(row["total_loss_usd"])

ev_df = pd.DataFrame([
    {"event": k, "avg_loss_musd": np.mean(v)/1e6, "count": len(v)}
    for k, v in ev_loss.items()
]).sort_values("avg_loss_musd", ascending=True)

bars = ax4.barh(ev_df["event"], ev_df["avg_loss_musd"],
                color="#E67E22", edgecolor="white", height=0.62)
for bar, cnt in zip(bars, ev_df["count"]):
    ax4.text(bar.get_width()+0.05, bar.get_y()+bar.get_height()/2,
             f"n={cnt:,}", va="center", fontsize=9)
ax4.set_xlabel("Avg Loss When Event Occurs ($M)", fontsize=11)
ax4.set_title("Disruption Event Impact\n(avg loss & frequency)", fontsize=12, fontweight="bold")

plt.suptitle("GlobalChain Monte Carlo Disruption Simulation — 10,000 Scenarios",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("disruption_simulation.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: disruption_simulation.png")


###  Task E — Your Turn!

1. **Mitigation ROI:** GlobalChain considers investing **$2M in resilience**
   (dual sourcing, safety stock). This would:
   - Reduce "Supplier Bankruptcy" probability: 10% → 4%
   - Reduce "Port Strike" impact by 20%

   Re-run the simulation with mitigation. Calculate expected savings and ROI.

2. **"Perfect Storm" stress test:** Force ALL events to occur simultaneously.
   What is the maximum possible loss? Is GlobalChain insured adequately?

3. **Sensitivity / Tornado Chart:** Vary each event's probability ±50% (one at a time)
   and measure the change in VaR 95%. Which event drives the most risk?

```python
# TODO — Task E code here
# Hint Q1:
# Modify EVENTS list → change prob/impact → re-run sim loop
# ROI = (mean_loss_before - mean_loss_after) / 2_000_000
```


---
## PART F — Power BI Export

Export all analysis results to a structured Excel workbook for Power BI.
Each sheet becomes a table you can model in Power BI Desktop.

### Power BI Dashboard Requirements (4 pages):
| Page | Title | Key Visuals |
|------|-------|-------------|
| 1 | Executive Risk Overview | KPI cards, risk matrix scatter, donut by priority |
| 2 | Supplier Risk Intelligence | Map, treemap (spend × tier), top-10 table |
| 3 | Demand & Forecast | Line chart with MC bands, seasonality column chart |
| 4 | Disruption Simulation | Loss histogram, exceedance curve, VaR gauge |


In [ ]:
# ════════════════════════════════════════════════════
#  PART F — Export to Power BI Excel Workbook
# ════════════════════════════════════════════════════

OUTPUT = "GlobalChain_PowerBI_Data.xlsx"

with pd.ExcelWriter(OUTPUT, engine="openpyxl") as writer:

    # Sheet 1 — Supplier Risk
    sup_export = df[[
        "supplier_id","name","country","region","annual_spend_musd",
        "lead_time_days","otd_pct","reject_rate_pct","financial_score",
        "geo_risk_score","esg_score","single_source","has_backup",
        "composite_risk","risk_tier"
    ]].copy()
    sup_export.to_excel(writer, sheet_name="Supplier_Risk", index=False)

    # Sheet 2 — Risk Register
    risk_register.to_excel(writer, sheet_name="Risk_Register", index=False)

    # Sheet 3 — Demand History
    demand_df.to_excel(writer, sheet_name="Demand_History", index=False)

    # Sheet 4 — Demand Forecast
    forecast_summary.to_excel(writer, sheet_name="Demand_Forecast", index=False)

    # Sheet 5 — MC Simulation (sample 2000 rows for Power BI performance)
    sim_export = sim_df.sample(2000, random_state=42).copy()
    sim_export["total_loss_musd"] = (sim_export["total_loss_usd"]/1e6).round(3)
    sim_export[["scenario_id","n_events","total_loss_musd",
                "loss_pct_rev","service_level","events","disrupted"]
               ].to_excel(writer, sheet_name="MC_Simulation", index=False)

    # Sheet 6 — KPI Summary
    kpi_df = pd.DataFrame({
        "KPI": [
            "Total Suppliers","Critical Risk Suppliers (Tier 4)",
            "Suppliers Without Backup","Single-Source Suppliers",
            "Avg Composite Risk Score","Top Risk (Risk Register)",
            "# High Priority Risks","Expected Annual Loss",
            "VaR 95%","CVaR 95%","Avg Service Level (MC)",
            "P(Service Level < 95%)"
        ],
        "Value": [
            len(suppliers_df),
            int((df["tier_num"]==3).sum()),
            int((~suppliers_df["has_backup"]).sum()),
            int(suppliers_df["single_source"].sum()),
            round(df["composite_risk"].mean(),2),
            risk_register.sort_values("risk_score",ascending=False)["risk_name"].iloc[0],
            int((risk_register["priority"]=="High").sum()),
            f"${sim_df['total_loss_usd'].mean()/1e6:.2f}M",
            f"${np.percentile(sim_df['total_loss_usd'],95)/1e6:.2f}M",
            f"${sim_df[sim_df['total_loss_usd']>=np.percentile(sim_df['total_loss_usd'],95)]['total_loss_usd'].mean()/1e6:.2f}M",
            f"{sim_df['service_level'].mean():.1f}%",
            f"{(sim_df['service_level']<95).mean():.1%}"
        ]
    })
    kpi_df.to_excel(writer, sheet_name="KPI_Summary", index=False)

    # Sheet 7 — Network Centrality
    centrality_df.to_excel(writer, sheet_name="Network_Centrality", index=False)

print(f"Excel workbook saved: {OUTPUT}")
print(f"Sheets: Supplier_Risk | Risk_Register | Demand_History | Demand_Forecast")
print(f"        MC_Simulation | KPI_Summary | Network_Centrality")
print(f"\nPower BI steps:")
print(f"  1. Power BI Desktop  ->  Get Data  ->  Excel Workbook")
print(f"  2. Load all 7 sheets as tables")
print(f"  3. Create relationships (Supplier_Risk region links to MC_Simulation)")
print(f"  4. Build 4-page dashboard per requirements above")


###  Detailed Power BI Task Guide

---

#### Page 1 — Executive Risk Overview
- **4 KPI Cards:** `Total Suppliers`, `Critical Risk Suppliers`, `Expected Annual Loss`, `VaR 95%`
- **Scatter plot** (Risk_Register): X = Impact, Y = Likelihood, size = risk_score, colour = priority
- **Donut chart** (Risk_Register): Count of risks by priority (Low/Medium/High)
- **Slicer:** Filter by risk category
- **Table:** Top 5 risks by score with current control

#### Page 2 — Supplier Risk Intelligence
- **Map** (Supplier_Risk): Bubble per country, size = annual_spend_musd, colour = risk_tier
- **Clustered bar chart:** Count of suppliers per risk tier, grouped by region
- **Treemap:** annual_spend_musd as size, risk_tier as colour, name as label
- **Table:** Top 10 highest composite_risk suppliers with drill-through
- **Slicers:** Region, risk_tier, single_source

#### Page 3 — Demand & Forecast
- **Line chart** (Demand_History + Demand_Forecast): Actual demand + MC_P5/P25/P75/P95 bands
- **Column chart:** Average demand by month number (seasonality pattern)
- **KPI Cards:** Mean forecast, uncertainty range (P95-P5), next-month forecast
- **Conditional formatting:** Highlight months where demand < P25 in red

#### Page 4 — Disruption Simulation
- **Histogram** (MC_Simulation): Distribution of total_loss_musd
- **Line chart:** Loss exceedance curve (sort descending, plot cumulative %)
- **Gauge visual:** Mean service_level vs 95% target
- **Bar chart:** Average loss grouped by events
- **KPI Card:** Count of scenarios where total_loss_musd > 10

---

> **Submission:** Export Power BI as PDF + submit the completed `.ipynb` notebook.


---
##  Reflection Questions

Answer in your written report (500–800 words total):

1. **Network vulnerability:** Which finding surprised you most? How should GlobalChain
   restructure its network to reduce its single point-of-failure exposure?

2. **ML vs expert judgement:** The K-Means algorithm may classify suppliers differently
   from a human expert. What are the benefits and risks of using ML for supplier
   risk profiling in real organisations?

3. **Communicating uncertainty:** How would you explain the Monte Carlo fan chart to a
   non-technical CFO? What is the single most important message?

4. **Resilience investment:** Write a one-paragraph **board recommendation** based on
   your simulation results. Which disruption should GlobalChain prioritise mitigating,
   and why?

5. **Model limitations:** List 3 assumptions in this simulation that could be improved
   with real data. What data would you collect, and how?

---

*GlobalChain Crisis Lab — Risk Management & Resilience in Logistics & Supply Chains*
*Complete all TODO tasks and submit: (1) this notebook (.ipynb) + (2) Power BI PDF*
